# Working

**Request:** [T09] Margin decline among Q2 churners

> New email from Sai Suram <sai@agents.agentstore.it.com>
> Subject: [T09] Margin decline among Q2 churners
> Thread ID: AAQkADI1N2Y5MTE3LTE1MDctNGY0Yy1iYzQ5LWEzNmE5NzAyYzk4NQAQAJHwnEGQeltGnvb5521TqRQ=
> 
> ##

Each cell below is one run in the sandbox, in the order it happened, with whatever it printed and produced.


In [1]:
import pandas as pd
import io

data = """
Customer,ChurnQuarter,OrderQuarter,Category,Margin,Refunded
C1,Q2,Q1,Hardware,4200,no
C1,Q2,Q2,Hardware,1800,no
C1,Q2,Q1,Software,2600,no
C1,Q2,Q2,Software,2400,no
C2,Q2,Q1,Hardware,3100,no
C2,Q2,Q2,Hardware,900,no
C2,Q2,Q1,Services,5000,no
C2,Q2,Q2,Services,1200,no
C2,Q2,Q2,Services,3300,yes
C3,Q3,Q1,Hardware,8000,no
C3,Q3,Q2,Hardware,200,no
C4,Q2,Q1,Software,1500,no
C4,Q2,Q2,Software,1100,no
"""

df = pd.read_csv(io.StringIO(data))

# Filter for customers who churned in Q2
q2_churners = df[df['ChurnQuarter'] == 'Q2']

# Exclude refunded orders
filtered_df = q2_churners[q2_churners['Refunded'] == 'no']

# Group by Customer, Category, and OrderQuarter to sum margins
# Pivot table to get Q1 and Q2 margins side-by-side for each customer-category combo
pivoted_df = filtered_df.pivot_table(index=['Customer', 'Category'], columns='OrderQuarter', values='Margin', aggfunc='sum').reset_index()
pivoted_df.columns.name = None # Remove the name of the columns index

# Fill NaN values with 0 for categories that might not have orders in both quarters
pivoted_df['Q1'] = pivoted_df['Q1'].fillna(0)
pivoted_df['Q2'] = pivoted_df['Q2'].fillna(0)

# Calculate margin decline
pivoted_df['Margin_Decline'] = pivoted_df['Q1'] - pivoted_df['Q2']

# Group by Category to get total margin decline per category among Q2 churners
category_decline = pivoted_df.groupby('Category')['Margin_Decline'].sum().reset_index()

# Find the category with the largest total margin decline
largest_decline_category = category_decline.loc[category_decline['Margin_Decline'].idxmax()]

output_filename = '/tmp/output/q2_churn_margin_decline_analysis.xlsx'

with pd.ExcelWriter(output_filename, engine='xlsxwriter') as writer:
    df.to_excel(writer, sheet_name='Raw Data', index=False)
    q2_churners.to_excel(writer, sheet_name='Q2 Churners', index=False)
    filtered_df.to_excel(writer, sheet_name='Filtered Data', index=False)
    pivoted_df.to_excel(writer, sheet_name='Customer Category Margins', index=False)
    category_decline.to_excel(writer, sheet_name='Category Decline Summary', index=False)

    # Add summary to the Category Decline Summary sheet
    worksheet = writer.sheets['Category Decline Summary']
    worksheet.write('A10', 'Key Finding:')
    worksheet.write('A11', f"The product category with the largest total margin decline from Q1 to Q2 among Q2 churners is {largest_decline_category['Category']} with a decline of {largest_decline_category['Margin_Decline']:.2f}.")

print(f"Largest margin decline category: {largest_decline_category['Category']}")
print(f"Decline amount: {largest_decline_category['Margin_Decline']}")


Largest margin decline category: Hardware
Decline amount: 4600



[files written: q2_churn_margin_decline_analysis.xlsx]
